# Frozen Repeat Near-Equatorial Orbit Design
## Mathematical Foundations of `design_frozen_repeat_joint.py`

This notebook derives every equation used in the joint SMA + eccentricity optimiser from first principles.
No prior knowledge of Orekit is assumed; the pure-Python cells run without a JVM.
Cells marked **[Orekit required]** call the actual propagator and need the environment set up.

### What the script does (one paragraph)
Given a near-equatorial LEO with a **k:q repeat ground track** (k satellite revolutions
in q sidereal days), `design_frozen_repeat_joint.py` finds the **osculating** initial
conditions $(a, e_x, e_y)$ such that two conditions hold simultaneously under the J2+J3
force model:
1. The ascending-node longitudes repeat exactly after q sidereal days (**repeat closure**).
2. The mean eccentricity vector stays near the J3 frozen fixed point — the orbit does
   not drift toward a circular (and therefore thermally / drag-variable) trajectory
   (**frozen eccentricity**).

The analytical theory gives good mean-element seeds.  The joint iteration corrects the
osculating initial conditions so a real J2+J3 propagator honours both conditions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle, FancyArrowPatch

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})

# ── EGM96 / WGS84 physical constants ─────────────────────────────────────────
MU          = 3.986004418e14    # m³/s²   gravitational parameter
RE          = 6_378_137.0       # m       equatorial radius
J2          =  1.08262668e-3    # dimensionless
J3          = -2.5415e-6        # dimensionless  (negative!)
OMEGA_EARTH =  7.2921150e-5     # rad/s   Earth sidereal rotation rate
T_SIDEREAL  = 86_164.0905       # s       one sidereal day

# ── Convenience functions (mirror src/core/frozen_orbit.py) ──────────────────
def mean_motion(a):       return np.sqrt(MU / a**3)
def orbital_period(a):    return 2*np.pi / mean_motion(a)

def raan_rate(a, e, i):
    """J2 secular dΩ/dt [rad/s].  Negative for prograde orbits."""
    n = mean_motion(a)
    return -(3/2)*n*J2*(RE/a)**2*np.cos(i)/(1-e**2)**2

def aop_rate(a, e, i):
    """J2 secular dω/dt [rad/s]."""
    n = mean_motion(a)
    return (3/4)*n*J2*(RE/a)**2*(5*np.cos(i)**2-1)/(1-e**2)**2

def frozen_ecc(a, i):
    """Coffey-Deprit frozen eccentricity (ω_f = 90°)."""
    return -(J3/(2*J2))*(RE/a)*np.sin(i)

print('Constants loaded.')

---
## Part 1 — Why Frozen Repeat Orbits?

### 1.1 The operational motivation

Earth-observation satellites need two properties that pull in different directions:

| Property | Engineering need | Orbital constraint |
|---|---|---|
| Predictable revisit | Fixed ground-track corridors | **Repeat ground track** |
| Stable altitude / geometry | No secular perigee drift | **Frozen eccentricity** |

A *repeat ground track* orbit satisfies
$$\frac{k}{q} = \frac{n_{\rm eff}}{\omega_E - \dot{\Omega}}$$
where $k$ is the number of satellite revolutions and $q$ the number of sidereal days in
one repeat cycle.  The left-hand side is a rational number chosen by the mission designer.

A *frozen orbit* satisfies the secular equilibrium of the eccentricity vector under
J3 forcing, preventing the argument of perigee from drifting and the perigee altitude
from oscillating with a long (beat) period of months to years.

### 1.2 Why both conditions must be satisfied simultaneously

- Correcting $a$ for ground-track repeat changes $n_{\rm eff}$ and therefore $\dot{\omega}$,
  which shifts the J3 equilibrium point slightly.
- Correcting the eccentricity vector changes $e$ and therefore $\dot{\Omega}$, which
  shifts the required SMA for ground-track repeat.

The coupling is weak ($O(J_2^2)$) but not negligible for precision orbit design.
The **joint iteration** (Section 9) resolves it.

---
## Part 2 — Geopotential and Secular Perturbation Rates

### 2.1 Geopotential zonal harmonics

The Earth's gravitational potential expanded in zonal spherical harmonics is
$$U = \frac{\mu}{r}\left[1 - \sum_{n=2}^{N} J_n \left(\frac{R_E}{r}\right)^n P_n(\sin\phi)\right]$$
where $P_n$ are Legendre polynomials, $\phi$ is geocentric latitude, and $J_n$ are
dimensionless coefficients encoding how much the Earth deviates from a sphere at
harmonic degree $n$.  The dominant terms for LEO are:

| Harmonic | Value | Physical meaning |
|---|---|---|
| $J_2 = +1.083\times10^{-3}$ | Equatorial bulge (oblateness) | Secular RAAN + ω̇ |
| $J_3 = -2.54\times10^{-6}$ | North–south asymmetry | Secular eccentricity drift |

$J_2$ is ~430× larger than $J_3$.

### 2.2 J2 secular rates (Brouwer 1959)

Averaging the equations of motion over one orbit (removing short-period terms), the
secular rates of the slow orbital elements under $J_2$ alone are:

**RAAN precession**
$$\dot{\Omega} = -\frac{3}{2}\frac{n J_2 R_E^2}{a^2(1-e^2)^2}\cos i \qquad [\text{rad/s}]$$

**Argument-of-perigee precession**
$$\dot{\omega} = \frac{3}{4}\frac{n J_2 R_E^2}{a^2(1-e^2)^2}(5\cos^2 i - 1) \qquad [\text{rad/s}]$$

**Mean anomaly correction**
$$\dot{M} = n\left[1 + \frac{3 J_2 R_E^2}{4 a^2 (1-e^2)^2}\sqrt{1-e^2}(3\cos^2 i - 1)\right]$$

The *effective mean motion* combining $\dot{\omega}$ and $\dot{M}$ is
$$n_{\rm eff} = \dot{M} + \dot{\omega} = n(1+\gamma)$$
where
$$\gamma = \frac{3 J_2 R_E^2}{4 p^2}\left[\sqrt{1-e^2}(3\cos^2 i-1) + (5\cos^2 i-1)\right], \quad p = a(1-e^2)$$

Note the two bracket terms: the first comes from secular $\dot{M}$ (Brouwer) and the
second from secular $\dot{\omega}$.  They both enter the repeat condition because the
repeat cycle counts complete traversals of the *argument of latitude* $u = \omega + M$.

In [ ]:
# Secular rates for the design example (from config.yaml neqfro section)
i_deg = 10.0           # near-equatorial
i_rad = np.radians(i_deg)
e_seed = 0.001

# Analytical SMA from 44:3 repeat at i=10°
k, q = 44, 3.0

# Keplerian seed
n0   = k * OMEGA_EARTH / q
a0   = (MU / n0**2)**(1/3)
alt0 = (a0 - RE)/1e3
T0   = orbital_period(a0)

# J2 secular rates at this orbit
Omega_dot = raan_rate(a0, e_seed, i_rad)      # rad/s
omega_dot = aop_rate(a0, e_seed, i_rad)        # rad/s

T_beat = 2*np.pi / abs(omega_dot)              # beat period (frozen->drifting->frozen)

print(f'Semi-major axis (Keplerian seed) : {a0/1e3:.3f} km   ({alt0:.1f} km altitude)')
print(f'Orbital period                   : {T0/60:.3f} min')
print(f'J2 RAAN rate  dΩ/dt             : {np.degrees(Omega_dot)*86400:.4f} deg/day')
print(f'J2 perigee rate dω/dt           : {np.degrees(omega_dot)*86400:.4f} deg/day')
print(f'Beat period  T_beat = 2π/|ω̇|   : {T_beat/86400:.2f} days  ({T_beat/T0:.0f} orbits)')

# Frozen eccentricity at this inclination
e_f = frozen_ecc(a0, i_rad)
print(f'Frozen eccentricity  e_f         : {e_f:.6e}')
print(f'J2 SP amplitude δe_sp ~ J2(RE/a)²: {J2*(RE/a0)**2:.3e}')
print(f'Ratio δe_sp / e_f               : {J2*(RE/a0)**2/e_f:.1f}×  (SP >> frozen at i=10°)')

---
## Part 3 — The Frozen Orbit Condition

### 3.1 Gauss variation-of-parameters for eccentricity

The Gauss VOP equations give the rate of change of the eccentricity vector
$(e_x, e_y) = (e\cos\omega,\; e\sin\omega)$ under a disturbing force.
Retaining only the secular part of the J3 perturbation (Kozai 1959, Coffey-Deprit 1986),
the secular evolution of $e$ and $\omega$ under J2+J3 is:

$$\dot{\omega}_{J2+J3} = \dot{\omega}_{J2} + \dot{\omega}_{J3}(e,i,\omega)$$
$$\dot{e}_{J3}(e,i,\omega) = -\frac{3nJ_3R_E^3}{2a^3(1-e^2)^3}\sin i\cos\omega$$

The J3 term drives $e$ toward zero when $\cos\omega > 0$ (ω near 90° or 270°) and away
from zero when $\cos\omega < 0$.  There is therefore a fixed point — the *frozen orbit* —
where the two effects balance.

### 3.2 The Coffey-Deprit formula

At the secular fixed point $\dot{e} = 0$ and $\dot{\omega} = 0$:

$$\boxed{e_f = -\frac{J_3}{2J_2}\frac{R_E}{a}\sin i, \qquad \omega_f = 90°\;(\text{or }270°)}$$

**Sign convention**: $J_3 < 0$ for Earth, so $e_f > 0$ for prograde orbits ($\sin i > 0$).

The two fixed points are:
- $\omega_f = 90°$: perigee above the north pole — **stable** (a small perturbation returns)
- $\omega_f = 270°$: perigee above the south pole — **unstable**

### 3.3 Physical picture

The J3 term is the *pear-shapedness* of Earth: the southern hemisphere is slightly more
bulged than the northern.  This creates an asymmetric tug on an eccentric orbit,
which, combined with the J2 perigee precession, produces a net torque that can be
balanced only at a specific eccentricity — the frozen value.

In [ ]:
# Frozen eccentricity as a function of inclination and altitude
i_arr  = np.linspace(0, 90, 300)    # deg
alt_km_list = [400, 550, 700]       # km

fig, ax = plt.subplots(figsize=(8, 4))
for alt in alt_km_list:
    a_alt = RE + alt*1e3
    ef    = frozen_ecc(a_alt, np.radians(i_arr))
    ax.plot(i_arr, ef, label=f'{alt} km')

ax.axvline(10, ls='--', color='gray', lw=1.2, label='Design point i=10°')
ax.axvline(51.6, ls=':', color='gray', lw=1.2, label='ISS i=51.6°')
ax.set_xlabel('Inclination [deg]')
ax.set_ylabel('Frozen eccentricity $e_f$')
ax.set_title('Coffey-Deprit frozen eccentricity vs. inclination and altitude')
ax.legend()
plt.tight_layout()
plt.show()

# At i=10° e_f is very small — this is the near-equatorial challenge
e_f_10  = frozen_ecc(a0, np.radians(10))
e_f_52  = frozen_ecc(a0, np.radians(51.6))
e_f_90  = frozen_ecc(a0, np.radians(90))
print(f'e_f at i=10°  : {e_f_10:.3e}   (near-equatorial)')
print(f'e_f at i=51.6°: {e_f_52:.3e}   (ISS-like)')
print(f'e_f at i=90°  : {e_f_90:.3e}   (polar)')

---
## Part 4 — The Near-Equatorial Challenge

### 4.1 Why $e_f \to 0$ as $i \to 0$

The frozen eccentricity scales as $\sin i$:
$$e_f = -\frac{J_3}{2J_2}\frac{R_E}{a}\sin i \xrightarrow{i\to 0} 0$$

At $i = 10°$, $e_f \sim 5\times10^{-5}$ — less than a hundredth of the J2 short-period
eccentricity oscillation amplitude:
$$\delta e_{\rm sp} \sim J_2\left(\frac{R_E}{a}\right)^2 \sim 10^{-3}$$

The raw osculating trajectory is dominated by $\delta e_{\rm sp}$, not by the secular
drift the optimiser should control.  This is why the Rosengren iteration uses the
**perigee-rotating frame** and the SCO uses a **running mean filter** — both techniques
separate the secular signal from the short-period noise.

### 4.2 Visualising the separation of scales

In [ ]:
# Synthetic eccentricity vector: frozen fixed point + short-period circle
# (illustrative model, not a full propagation)

t_arr     = np.linspace(0, 5*T0, 2000)     # 5 orbits
omega_dot_val = aop_rate(a0, e_f_10, i_rad) # rad/s

# J2 short-period amplitude (rough estimate: J2*(RE/a)^2)
A_sp = J2 * (RE/a0)**2  

# Short-period frequency ≈ n (one cycle per orbit)
n_val = mean_motion(a0)

# Synthetic osculating (ex, ey): frozen point + SP oscillation
ex_syn = e_f_10 * np.cos(omega_dot_val*t_arr) \
       + A_sp * np.cos(n_val*t_arr)
ey_syn = e_f_10 * np.sin(omega_dot_val*t_arr) \
       + A_sp * np.sin(n_val*t_arr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: inertial phase space
ax = axes[0]
ax.plot(ex_syn, ey_syn, lw=0.6, color='steelblue', label='Osculating')
ax.plot(0, 0, 'k+', ms=10)
ax.set_aspect('equal')
ax.set_xlabel('$e_x = e\\cos\\omega$')
ax.set_ylabel('$e_y = e\\sin\\omega$')
ax.set_title('Inertial frame (5 orbits)')
ax.annotate('Frozen\npoint', xy=(e_f_10, 0), xytext=(e_f_10*3, A_sp*0.7),
            arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato')
ax.legend(fontsize=9)

# Right: rotating frame
ex_rot = ex_syn * np.cos(-omega_dot_val*t_arr) - ey_syn * np.sin(-omega_dot_val*t_arr)
ey_rot = ex_syn * np.sin(-omega_dot_val*t_arr) + ey_syn * np.cos(-omega_dot_val*t_arr)

ax2 = axes[1]
ax2.plot(ex_rot, ey_rot, lw=0.6, color='seagreen', label='Osculating (rotating frame)')
ax2.plot(0, e_f_10, 'r*', ms=10, label='Frozen fixed point $(0, e_f)$')
ax2.set_aspect('equal')
ax2.set_xlabel('$\\xi_{rot}$')
ax2.set_ylabel('$\\eta_{rot}$')
ax2.set_title('Perigee-rotating frame')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Frozen eccentricity  e_f   = {e_f_10:.3e}')
print(f'SP oscillation amplitude A = {A_sp:.3e}   ({A_sp/e_f_10:.0f}× larger than e_f)')
print('In the rotating frame the frozen point is stationary at (0, e_f);')
print('the osculating trajectory circles tightly around it.')

---
## Part 5 — Repeat Ground-Track Condition

### 5.1 What repeating means geometrically

A ground track *repeats exactly* after $q$ sidereal days if and only if in that same
duration the satellite completes exactly $k$ revolutions **relative to the rotating Earth
referenced to the precessing node**.

In $q$ sidereal days:
- Earth rotates by $q \cdot 2\pi$ in the inertial frame.
- The RAAN advances by $\dot{\Omega}\cdot q T_\odot$ (negative for prograde).
- So Earth rotates by $(\omega_E - \dot{\Omega})\cdot q T_\odot$ relative to the
  node-fixed frame.
- The satellite traverses $n_{\rm eff}\cdot q T_\odot$ radians of argument of latitude.

For $k$ complete passes over the same ground track:
$$n_{\rm eff}\cdot q T_\odot = k\cdot 2\pi \qquad \text{and} \qquad
(\omega_E-\dot{\Omega})\cdot q T_\odot = q\cdot 2\pi$$

Dividing:
$$\boxed{\frac{n_{\rm eff}}{\omega_E - \dot{\Omega}} = \frac{k}{q}}$$

Solving for the required $n_{\rm eff}$:
$$n_{\rm eff} = \frac{k(\omega_E - \dot{\Omega})}{q}$$

Then $a$ follows from $n_{\rm eff} = n(1+\gamma)$ and Kepler's third law $a^3 = \mu/n^2$.

### 5.2 Why iteration is needed for the SMA

Both $\dot{\Omega}(a,e,i)$ and $\gamma(a,e,i)$ depend on $a$, so the equation
$$n(1+\gamma) = \frac{k(\omega_E-\dot{\Omega})}{q}$$
must be solved iteratively. A 5-iteration fixed-point loop converges to metre-level
accuracy (see `frozen_orbit.repeat_ground_track_sma`).

In [ ]:
def n_eff_factor(a, e, i):
    """Combined J2 secular correction γ: n_eff = n*(1+γ)."""
    eta2 = 1 - e**2
    p2   = (a*eta2)**2
    c2i  = np.cos(i)**2
    return (3*J2*RE**2/(4*p2)) * (np.sqrt(eta2)*(3*c2i-1) + (5*c2i-1))

def repeat_sma(k, q, i_rad, e=0.001, tol=1.0, max_iter=100):
    """J2-corrected SMA [m] for a k:q repeat orbit."""
    a = (MU / (k*OMEGA_EARTH/q)**2)**(1/3)
    for _ in range(max_iter):
        dOmega = raan_rate(a, e, i_rad)
        gamma  = n_eff_factor(a, e, i_rad)
        n_req  = k*(OMEGA_EARTH - dOmega) / q / (1 + gamma)
        a_new  = (MU/n_req**2)**(1/3)
        if abs(a_new - a) < tol:
            return a_new
        a = a_new
    return a

a_j2 = repeat_sma(k, q, i_rad, e=e_f_10)
a_2body = (MU / (k*OMEGA_EARTH/q)**2)**(1/3)
T_nodal = orbital_period(a_j2)

print('── Repeat ground-track SMA ─────────────────────────────────')
print(f'  Two-body seed         : {a_2body/1e3:.4f} km')
print(f'  J2-corrected (i={i_deg}°): {a_j2/1e3:.4f} km  (Δa = +{a_j2-a_2body:.1f} m)')
print(f'  Altitude (J2)         : {(a_j2-RE)/1e3:.2f} km')
print(f'  Orbital period        : {orbital_period(a_j2)/60:.4f} min')
print()
print('Why the J2 correction is positive for prograde near-equatorial orbits:')
print(f'  dΩ/dt = {np.degrees(raan_rate(a_j2,e_f_10,i_rad))*86400:.4f} deg/day (retrograde nodal drift)')
print('  Retrograde drift makes (ωE - Ω̇) > ωE → need larger n_eff → smaller a... ')
print('  ...BUT γ is positive and larger than the Ω̇ correction → net a increases.')

---
## Part 6 — Mean vs. Osculating Elements

### 6.1 The distinction

*Mean elements* are the constants of integration of the averaged (secular) equations of
motion — they change only on long timescales (days to years).  The Coffey-Deprit
frozen condition and the J2 repeat SMA formula are both expressed in mean elements.

*Osculating elements* are the instantaneous Keplerian elements that describe the
two-body orbit tangent to the true trajectory at each moment.  They contain:
- **Short-period** variations (period ≈ $T_{\rm orb}$)
- **Long-period** variations (period ≈ $T_{\rm beat} = 2\pi/|\dot{\omega}|$)
- **Secular** drift (the mean-element piece)

The numerical propagator in `j3_propagator.py` initialises from osculating elements.

### 6.2 The offset between mean and osculating initial conditions

If we initialise the propagator at the **mean** frozen eccentricity $(0, e_f)$, the
short-period terms immediately shift the osculating state away from the frozen point.
After half a beat period the mean element (in the inertial frame) has rotated, and the
osculating state is far from $(0, e_f)$.

We need to find the **osculating** initial condition $(e_{x,\rm osc}, e_{y,\rm osc})$
whose *time-averaged* rotating-frame trajectory returns $(0, e_f)$ as the mean.
This is what the Rosengren iteration computes.

### 6.3 Why the offset matters for the SMA too

The same issue applies to the SMA: the analytical $a$ satisfies the repeat condition
for mean elements.  The osculating $a$ differs by the J2 short-period correction to
the semi-major axis:
$$\delta a_{\rm sp} \sim J_2 \frac{R_E^2}{a} \sim 100\text{ m}$$

The SMA Newton step in the joint iteration corrects for this numerically.

---
## Part 7 — The Perigee-Rotating Frame

### 7.1 Why the inertial frame fails for averaging

Under J2, the argument of perigee precesses secularly at $\dot{\omega}$.  In the
*inertial* (ex, ey) frame the frozen orbit's mean eccentricity vector rotates at
the beat rate:
$$\bar{\mathbf{e}}_{\rm inertial}(t) = e_f\begin{pmatrix}\cos(\omega_0 + \dot{\omega}t)\\
\sin(\omega_0 + \dot{\omega}t)\end{pmatrix}$$

The beat period is $T_{\rm beat} = 2\pi/|\dot{\omega}|$.  For our design orbit at $i=10°$
this is on the order of tens of years. But even for a mid-inclination orbit the beat
period may be days to months.

If we average the inertial (ex, ey) over a window shorter than one beat period, the
result is biased away from $(0, e_f)$ — it depends on where in the beat cycle the
averaging window sits.  **The inertial time-average is not a reliable estimator
of the frozen fixed point.**

### 7.2 The rotating-frame construction

Define the **perigee-rotating frame** by subtracting the secular J2 precession from the
argument of perigee at each instant:
$$\xi_{\rm rot}(t) = e(t)\cos\bigl(\omega(t) - \dot{\omega}\,t\bigr)$$
$$\eta_{\rm rot}(t) = e(t)\sin\bigl(\omega(t) - \dot{\omega}\,t\bigr)$$

where $\dot{\omega}$ is the *secular* J2 precession rate (a constant computed once from
the analytical elements).  In this frame:

- The secular mean-element frozen fixed point is **stationary at $(0, e_f)$** for
  all time.
- The osculating trajectory still shows short-period oscillations (period $\approx T_{\rm orb}$)
  but they are centred on the fixed point.
- The **time-average of the rotating-frame trajectory converges to $(0, e_f)$** for
  any averaging window of a few repeat cycles, regardless of the beat period.

### 7.3 Why t = 0 is special for the correction rule

At $t = 0$ the rotating frame coincides with the inertial frame (the precession angle
$\dot{\omega}\cdot 0 = 0$).  Therefore the **correction to the inertial initial
condition equals the rotating-frame error**:
$$\Delta(e_x, e_y)_{t=0,\;\rm inertial} = \bigl(e_{x,\rm target} - \langle\xi_{\rm rot}\rangle,\;
e_{y,\rm target} - \langle\eta_{\rm rot}\rangle\bigr)$$

This identity is the core of the Rosengren correction rule.

In [ ]:
# Demonstrate the beat period and frame rotation (synthetic model)

omega_dot_val = aop_rate(a_j2, e_f_10, i_rad)   # rad/s
T_beat_s      = 2*np.pi / abs(omega_dot_val)
n_val         = mean_motion(a_j2)

# Simulate over 1 beat period
t_beat = np.linspace(0, T_beat_s, 20_000)

A_sp  = J2*(RE/a_j2)**2   # rough SP amplitude

# Synthetic osculating state: frozen mean + SP oscillation + long-period circle
# (LP period ≈ T_beat; amplitude ~ A_sp * (e_f/e_sp) is negligible here)
ex_inertial = e_f_10*np.cos(omega_dot_val*t_beat) + A_sp*np.cos(n_val*t_beat)
ey_inertial = e_f_10*np.sin(omega_dot_val*t_beat) + A_sp*np.sin(n_val*t_beat)

# Rotating-frame transform
cos_rot = np.cos(-omega_dot_val*t_beat)
sin_rot = np.sin(-omega_dot_val*t_beat)
xi_rot  =  ex_inertial*cos_rot - ey_inertial*sin_rot
eta_rot =  ex_inertial*sin_rot + ey_inertial*cos_rot

# Compare averaging in the two frames over windows of increasing length
T_orb_s = orbital_period(a_j2)
windows = [1, 3, 10, 30]
print(f'T_beat = {T_beat_s/86400:.1f} days = {T_beat_s/T_orb_s:.0f} orbits')
print(f'T_orb  = {T_orb_s/60:.2f} min')
print()
print(f'{"Window [orbits]":>20s} | {"⟨ex⟩ inertial":>15s} {"⟨ey⟩ inertial":>15s}'
      f' | {"⟨ξ⟩ rotating":>14s} {"⟨η⟩ rotating":>14s}')
print('-'*85)
for w in windows:
    mask = t_beat <= w*T_orb_s
    ei_x = np.mean(ex_inertial[mask])
    ei_y = np.mean(ey_inertial[mask])
    er_x = np.mean(xi_rot[mask])
    er_y = np.mean(eta_rot[mask])
    print(f'{w:>20d} | {ei_x:>+15.4e} {ei_y:>+15.4e} | {er_x:>+14.4e} {er_y:>+14.4e}')
print()
print(f'Target: (ex_target, ey_target) = (0, {e_f_10:.4e})')
print('Rotating-frame average converges to target within a few orbits.')
print('Inertial average is biased for windows shorter than T_beat.')

---
## Part 8 — The Rosengren Eccentricity Iteration

### 8.1 Problem statement

We want to find the **osculating** initial eccentricity vector $\mathbf{z}_0 = (e_{x,0}, e_{y,0})$
such that the propagated rotating-frame time-average equals the analytical target:
$$\langle\boldsymbol{\xi}_{\rm rot}(\mathbf{z}_0)\rangle_{T} = \mathbf{z}_{\rm target} = (0, e_f)$$

Define the **residual function**
$$\mathbf{F}(\mathbf{z}_0) = \mathbf{z}_{\rm target} - \langle\boldsymbol{\xi}_{\rm rot}(\mathbf{z}_0)\rangle_T$$

We seek $\mathbf{z}_0^*$ such that $\mathbf{F}(\mathbf{z}_0^*) = \mathbf{0}$.

### 8.2 The linear sensitivity argument

The map from initial condition to time-average is approximately linear near the
frozen fixed point.  To first order:
$$\langle\boldsymbol{\xi}_{\rm rot}(\mathbf{z}_0 + \Delta\mathbf{z})\rangle_T
\approx \langle\boldsymbol{\xi}_{\rm rot}(\mathbf{z}_0)\rangle_T + \mathbf{\Phi}\,\Delta\mathbf{z}$$

where $\mathbf{\Phi} = \partial\langle\boldsymbol{\xi}\rangle/\partial\mathbf{z}_0$ is a
$2\times2$ sensitivity matrix.  Near the frozen point $\mathbf{\Phi} \approx \mathbf{I}$
(the mean trajectory responds nearly one-for-one to the initial condition).  This is
the *key assumption* behind the Rosengren iteration.

### 8.3 Correction rule and convergence

Substituting $\mathbf{\Phi} \approx \mathbf{I}$ into a fixed-point iteration:
$$\mathbf{z}_0^{(j+1)} = \mathbf{z}_0^{(j)} + \mathbf{F}(\mathbf{z}_0^{(j)})
= \mathbf{z}_0^{(j)} + \bigl(\mathbf{z}_{\rm target} - \langle\boldsymbol{\xi}_{\rm rot}^{(j)}\rangle_T\bigr)$$

In `rosengren.py` / `joint_optimizer.py` this is:
```python
d_ex = ex_target - ex_avg
d_ey = ey_target - ey_avg
ex_c += d_ex
ey_c += d_ey
```

**Convergence** is geometric with rate $\|\mathbf{I} - \mathbf{\Phi}\|$. Because
$\mathbf{\Phi}$ is close to identity, the residual halves each iteration —
typically ~50 iterations are needed to reach $10^{-9}$ (the tolerance in config.yaml).

### 8.4 Why $\mathbf{\Phi} \approx \mathbf{I}$

The short-period corrections to the time-average are $O(J_2)$.  The perturbation to
$\Phi$ away from identity is therefore $O(J_2) \sim 10^{-3}$ — the fixed-point
iteration converges geometrically with rate $\sim 0.001$ per iteration near the frozen
point, which is consistent with the fast empirical convergence observed in practice.

In [ ]:
# Demonstrate the Rosengren iteration on the synthetic model
# (self-contained — no Orekit needed)

def synthetic_mean(ex_init, ey_init, omega_dot, n_orb, T_orb, A_sp):
    """Return rotating-frame time-average for the synthetic model.
    
    Models: osc(t) = mean_init * exp(i*omega_dot*t) + A_sp * exp(i*n*t)
    Then derotates by omega_dot*t and averages."""
    n    = 2*np.pi / T_orb
    t    = np.linspace(0, n_orb*T_orb, n_orb*500)
    ex_i = ex_init*np.cos(omega_dot*t) - ey_init*np.sin(omega_dot*t) + A_sp*np.cos(n*t)
    ey_i = ex_init*np.sin(omega_dot*t) + ey_init*np.cos(omega_dot*t) + A_sp*np.sin(n*t)
    # Derotate
    xi  = ex_i*np.cos(-omega_dot*t) - ey_i*np.sin(-omega_dot*t)
    eta = ex_i*np.sin(-omega_dot*t) + ey_i*np.cos(-omega_dot*t)
    return np.mean(xi), np.mean(eta)

# Run Rosengren iteration
n_orb   = 5
T_orb_s = orbital_period(a_j2)
A_sp    = J2*(RE/a_j2)**2
odot    = omega_dot_val

ex_target, ey_target = 0.0, e_f_10   # frozen target in rotating frame

# Initial guess: analytical mean target
ex_c, ey_c = ex_target, ey_target

residuals = []
for k_it in range(12):
    xi_avg, eta_avg = synthetic_mean(ex_c, ey_c, odot, n_orb, T_orb_s, A_sp)
    d_ex = ex_target - xi_avg
    d_ey = ey_target - eta_avg
    res  = np.hypot(d_ex, d_ey)
    residuals.append(res)
    ex_c += d_ex
    ey_c += d_ey

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(range(1, len(residuals)+1), residuals, 'o-', color='steelblue')
ax.set_xlabel('Rosengren iteration')
ax.set_ylabel('Residual $|\\Delta(e_x, e_y)|$')
ax.set_title('Rosengren iteration convergence (synthetic model)')
plt.tight_layout()
plt.show()

print(f'Converged osculating initial condition:')
print(f'  (ex_c, ey_c) = ({ex_c:.6e}, {ey_c:.6e})')
print(f'  Analytical mean target : ({ex_target:.6e}, {ey_target:.6e})')
print(f'  Difference (SP offset) : ({ex_c-ex_target:.3e}, {ey_c-ey_target:.3e})')

---
## Part 9 — The SMA Newton Step (Ground-Track Closure)

### 9.1 Measuring closure

After propagating for one repeat cycle ($q$ sidereal days), the satellite's ascending
node longitude is compared with the first crossing.  For a perfectly repeating orbit
the **closure** $\Delta\lambda_{\rm AN}$ is zero.

Closure is measured in `analysis.ascending_node_longitudes` by detecting south-to-north
equatorial crossings and linearly interpolating to sub-step accuracy.

### 9.2 Analytical Jacobian $d(\Delta\lambda_{\rm AN})/da$

The ascending-node longitude drift per orbit is:
$$\Delta\lambda_{\rm AN/orb} = (\dot{\Omega} - \omega_E)\,T_{\rm nodal}$$

After $k$ orbits the total closure is:
$$\Delta\lambda_{\rm total} = k\,(\dot{\Omega} - \omega_E)\,T_{\rm nodal}$$

Differentiating with respect to $a$:
$$\frac{d\,\Delta\lambda_{\rm total}}{da} = k\left[\frac{d\dot{\Omega}}{da}\,T_{\rm nodal}
+ (\dot{\Omega} - \omega_E)\,\frac{dT_{\rm nodal}}{da}\right]$$

Using $\dot{\Omega} \propto n\cdot a^{-2} \propto a^{-7/2}$:
$$\frac{d\dot{\Omega}}{da} = -\frac{7}{2}\frac{\dot{\Omega}}{a}$$

Using $T_{\rm nodal} \approx 2\pi/n \propto a^{3/2}$:
$$\frac{dT_{\rm nodal}}{da} = \frac{3}{2}\frac{T_{\rm nodal}}{a}$$

Substituting:
$$\frac{d\,\Delta\lambda_{\rm total}}{da}
= k\,\frac{T_{\rm nodal}}{a}\left[-\frac{7}{2}\dot{\Omega}
+ \frac{3}{2}(\dot{\Omega} - \omega_E)\right]
= k\,\frac{T_{\rm nodal}}{a}\left(-2\dot{\Omega} - \frac{3}{2}\omega_E\right)$$

which matches exactly the code in `joint_optimizer.py`:
```python
J_a = (k_orbits * T_nodal / a) * (-2 * nodal_rate - 1.5 * OMEGA_EARTH)
da  = -np.radians(closure_deg) / J_a
```

### 9.3 Sign and magnitude

For a prograde near-equatorial orbit:
- $\dot{\Omega} < 0$ (retrograde RAAN drift)
- $\omega_E > 0$
- $J_a = (k T_{\rm nodal}/a)(-2\dot{\Omega} - 1.5\omega_E)$
  $= (k T_{\rm nodal}/a)(+|2\dot{\Omega}| - 1.5\omega_E)$

For our 44:3 orbit at $i=10°$ the nodal drift is small, so $J_a < 0$, meaning
increasing $a$ decreases the closure error — a positive closure (ground track drifts
eastward) requires a smaller $a$ (faster satellite).

In [ ]:
# Verify the analytical Jacobian numerically
# (pure math — no propagation, just rate formulas)

a_des = a_j2
e_des = e_f_10

# Analytical Jacobian
nodal_rate_val = raan_rate(a_des, e_des, i_rad)
T_nodal_val    = orbital_period(a_des)            # ≈ T_nodal
J_a_analytic   = (k * T_nodal_val / a_des) * (-2*nodal_rate_val - 1.5*OMEGA_EARTH)

# Numerical Jacobian: finite difference of closure = k*(Ω̇ - ωE)*T_nodal
def closure_model(a_test):
    dOm = raan_rate(a_test, e_des, i_rad)
    T   = orbital_period(a_test)
    return k * (dOm - OMEGA_EARTH) * T    # radians

da_fd  = 1.0    # 1 m perturbation
J_fd   = (closure_model(a_des + da_fd) - closure_model(a_des - da_fd)) / (2*da_fd)

print('SMA Newton step Jacobian d(closure_rad)/da')
print(f'  Analytical formula  : {J_a_analytic:.6e} rad/m')
print(f'  Finite difference   : {J_fd:.6e} rad/m')
print(f'  Relative error      : {abs(J_a_analytic-J_fd)/abs(J_fd):.2e}')
print()

# Example Newton step for a synthetic 0.1° closure error
closure_err_deg  = 0.1      # degrees
da_newton = -np.radians(closure_err_deg) / J_a_analytic
print(f'For a closure error of {closure_err_deg}°:')
print(f'  Newton step Δa = {da_newton:+.3f} m')
print(f'  (negative: must reduce a to pull ground track back westward)')

---
## Part 10 — The Joint Iteration

### 10.1 Coupling between the two sub-problems

| Variable changed | Effect on SMA sub-problem | Effect on eccentricity sub-problem |
|---|---|---|
| $a$ changed by $\Delta a$ | Closure directly depends on $a$ | $\dot{\omega}$ changes → rotating-frame fixed point shifts |
| $(e_x,e_y)$ changed | $\dot{\Omega}$ changes by $O(e)$ → slight closure shift | Directly corrects the mean eccentricity |

The coupling is first-order in $e$ and $J_2$, so it is small but not negligible for
high-fidelity design.

### 10.2 Sequential correction (Rosengren A34934 §IV)

Each outer iteration applies two sub-steps in order:

**Step A — SMA correction** (ground-track closure)
1. Propagate from current $(a, e_x, e_y, i, \Omega, M_0)$ for one repeat cycle
   using a coarse time step (adequate for AN-crossing detection).
2. Measure closure $\Delta\lambda_{\rm AN}$ [deg].
3. Apply Newton step: $a \leftarrow a - \Delta\lambda_{\rm AN,rad} / J_a$.

**Step B — Eccentricity correction** (Rosengren rotating-frame)
1. Propagate from current $(a, e_x, e_y, \ldots)$ for $n_{\rm cycles}$ repeat cycles
   using the fine time step.
2. Compute rotating-frame time-average $(\langle\xi_{\rm rot}\rangle, \langle\eta_{\rm rot}\rangle)$.
3. Apply residual: $(e_x, e_y) \leftarrow (e_x, e_y)
   + (e_{x,\rm target} - \langle\xi\rangle,\; e_{y,\rm target} - \langle\eta\rangle)$.

**Convergence check:** both $|\Delta\lambda_{\rm AN}| < \epsilon_{\rm closure}$ and
$\|(e_x, e_y)_{\rm residual}\| < \epsilon_{\rm ecc}$ must hold simultaneously.

### 10.3 Why sequential rather than simultaneous?

The coupling between $a$ and $(e_x, e_y)$ is $O(e)$.  Applying the corrections
alternately in one outer loop converges to the coupled solution without requiring a
coupled $3\times3$ Jacobian — a major computational saving since each propagation
call takes seconds.  The sequential scheme mirrors the original Rosengren approach
(Reference A34934 §IV-B).

### 10.4 Propagation count

Per outer iteration:
- 1 coarse propagation for SMA closure (one repeat cycle)
- 1 fine propagation for eccentricity averaging ($n_{\rm cycles}$ repeat cycles)

Total: $2 N_{\rm iter}$ propagations.  With $N_{\rm iter} \approx 20$ and $t_{\rm prop}
\approx 10$ s (J2+J3), the full design run takes $\sim 400$ s.

In [ ]:
# Illustrate the joint iteration logic with the analytical model (no Orekit)

def synthetic_closure(a, e, i_rad, k, T_sidereal):
    """Synthetic ground-track closure [deg] from secular rate model."""
    dOm = raan_rate(a, e, i_rad)
    T   = orbital_period(a)
    return np.degrees(k * (dOm - OMEGA_EARTH) * T)

def synthetic_ecc_avg(ex_init, ey_init, omega_dot, n_orb, T_orb, A_sp):
    """Rotating-frame time-average (synthetic model)."""
    return synthetic_mean(ex_init, ey_init, omega_dot, n_orb, T_orb, A_sp)

# Start from analytical seed
a_it   = a_j2 + 300.0           # perturb by +300 m to create a closure error
ex_it  = ex_target
ey_it  = ey_target

hist   = []
n_orb_ecc = 5

for it in range(15):
    e_it    = np.hypot(ex_it, ey_it)
    
    # ── Step A: SMA Newton step ────────────────────────────────────────────
    cl_deg  = synthetic_closure(a_it, e_it, i_rad, k, T_SIDEREAL)
    od_it   = raan_rate(a_it, e_it, i_rad)
    Tn_it   = orbital_period(a_it)
    Ja_it   = (k * Tn_it / a_it) * (-2*od_it - 1.5*OMEGA_EARTH)
    da_it   = -np.radians(cl_deg) / Ja_it
    a_it   += da_it
    
    # ── Step B: Rosengren eccentricity correction ──────────────────────────
    odot_it = aop_rate(a_it, e_it, i_rad)
    Torb_it = orbital_period(a_it)
    xi_avg, eta_avg = synthetic_ecc_avg(
        ex_it, ey_it, odot_it, n_orb_ecc, Torb_it, J2*(RE/a_it)**2
    )
    d_ex = ex_target - xi_avg
    d_ey = ey_target - eta_avg
    ex_it += d_ex
    ey_it += d_ey
    
    hist.append({
        'closure': cl_deg, 'da': da_it,
        'ecc_res': np.hypot(d_ex, d_ey),
        'a': a_it, 'ex': ex_it, 'ey': ey_it
    })

# Plot convergence
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
iters = range(1, len(hist)+1)
axes[0].semilogy(iters, [abs(h['closure']) for h in hist], 'o-', color='tomato', label='|closure| [deg]')
axes[0].set_ylabel('|Closure| [deg]')
axes[0].set_title('Joint iteration convergence (analytical model)')
axes[1].semilogy(iters, [h['ecc_res'] for h in hist], 's-', color='seagreen', label='Ecc residual')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Ecc residual')
for ax in axes: ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Converged SMA  : {hist[-1]["a"]/1e3:.4f} km  (analytical: {a_j2/1e3:.4f} km)')
print(f'Converged (ex) : {hist[-1]["ex"]:.5e}  (analytical target: {ex_target:.5e})')
print(f'Converged (ey) : {hist[-1]["ey"]:.5e}  (analytical target: {ey_target:.5e})')

---
## Part 11 — Full Worked Example (config.yaml Parameters)

This section replicates all analytical calculations from `design_frozen_repeat_joint.py`
without Orekit, so you can verify the numbers before running the full propagation.

The config (neqfro section):
- `orbit_type: near_equatorial`
- `inclination_deg: 10.0`
- `repeat_k: 44`, `repeat_q_days: 3.0`
- `aop_deg: 90.0`  (frozen condition)

In [ ]:
print('='*62)
print('  ANALYTICAL FROZEN-ORBIT ESTIMATES  [near_equatorial]')
print('='*62)

# Design parameters
i_deg_des  = 10.0
aop_deg    = 90.0
k_des, q_des = 44, 3.0
i_rad_des  = np.radians(i_deg_des)
aop_rad    = np.radians(aop_deg)

# Step 1: J2-corrected SMA
a_des   = repeat_sma(k_des, q_des, i_rad_des, e=0.001)

# Step 2: Frozen eccentricity at this SMA
e_f_des = frozen_ecc(a_des, i_rad_des)
ex_des  = e_f_des * np.cos(aop_rad)
ey_des  = e_f_des * np.sin(aop_rad)

# Derived quantities
T_orb_des   = orbital_period(a_des)
odot_des    = aop_rate(a_des, e_f_des, i_rad_des)
Odot_des    = raan_rate(a_des, e_f_des, i_rad_des)
T_beat_des  = 2*np.pi / abs(odot_des)
alt_des     = (a_des - RE)/1e3
T_repeat    = q_des * T_SIDEREAL

# Two-body comparison
a_2b = (MU / (k_des*OMEGA_EARTH/q_des)**2)**(1/3)

print(f'  Repeat cycle            : {k_des}:{q_des:.0f}  ({k_des} orbits / {q_des:.0f} sidereal days)')
print(f'  Inclination             : {i_deg_des:.1f} deg')
print(f'  Arg. of perigee (frozen): {aop_deg:.1f} deg')
print()
print(f'  SMA (two-body seed)     : {a_2b/1e3:.4f} km')
print(f'  SMA (J2-corrected)      : {a_des/1e3:.4f} km  (+{a_des-a_2b:.1f} m)')
print(f'  Altitude                : {alt_des:.2f} km')
print(f'  Orbital period          : {T_orb_des/60:.4f} min')
print(f'  Repeat cycle duration   : {T_repeat/3600:.4f} hr  = {T_repeat/86400:.4f} days')
print()
print(f'  Frozen eccentricity e_f : {e_f_des:.6e}')
print(f'  (ex, ey) seed           : ({ex_des:.5e}, {ey_des:.5e})')
print()
print(f'  J2 RAAN rate dΩ/dt      : {np.degrees(Odot_des)*86400:.4f} deg/day')
print(f'  J2 AoP rate  dω/dt      : {np.degrees(odot_des)*86400:.4f} deg/day')
print(f'  Beat period  T_beat     : {T_beat_des/86400:.2f} days  ({T_beat_des/T_orb_des:.0f} orbits)')
print()
print(f'  Short-period δe_sp ~ J2(RE/a)² : {J2*(RE/a_des)**2:.3e}')
print(f'  Signal-to-noise  e_f / δe_sp   : {e_f_des / (J2*(RE/a_des)**2):.3f}')
print('  (< 0.1 → rotating-frame averaging is essential)')
print('='*62)

In [ ]:
# Summary figure: all key quantities on one diagram

fig = plt.figure(figsize=(12, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# (a) e_f vs inclination
ax1 = fig.add_subplot(gs[0, 0])
i_v = np.linspace(0.5, 90, 300)
ax1.plot(i_v, frozen_ecc(a_des, np.radians(i_v)), color='steelblue')
ax1.axvline(i_deg_des, ls='--', color='tomato', lw=1.2, label=f'i={i_deg_des}°')
ax1.axhline(e_f_des, ls=':', color='tomato', lw=1.2)
ax1.set_xlabel('Inclination [deg]')
ax1.set_ylabel('$e_f$')
ax1.set_title('Frozen eccentricity')
ax1.legend(fontsize=8)

# (b) Beat period vs inclination
ax2 = fig.add_subplot(gs[0, 1])
odot_v   = aop_rate(a_des, 0.001, np.radians(i_v))
tbeat_v  = 2*np.pi / np.abs(odot_v)
ax2.plot(i_v, tbeat_v/86400, color='seagreen')
ax2.axvline(i_deg_des, ls='--', color='tomato', lw=1.2)
ax2.set_xlabel('Inclination [deg]')
ax2.set_ylabel('$T_{beat}$ [days]')
ax2.set_title('Beat period')
# Mark critical inclination (ω̇ = 0 at 5cos²i = 1 → i = 63.43°)
ax2.axvline(np.degrees(np.arccos(1/np.sqrt(5))), ls=':', color='gray', lw=1, label='Critical i')
ax2.set_ylim(0, 1000)
ax2.legend(fontsize=8)

# (c) Closure Jacobian J_a vs a
ax3 = fig.add_subplot(gs[0, 2])
a_v  = np.linspace(6600e3, 7000e3, 200)
Ja_v = np.array([(k_des * orbital_period(aa) / aa) *
                 (-2*raan_rate(aa, e_f_des, i_rad_des) - 1.5*OMEGA_EARTH)
                 for aa in a_v])
ax3.plot(a_v/1e3, Ja_v*1e6, color='darkorange')
ax3.axvline(a_des/1e3, ls='--', color='tomato', lw=1.2, label='Design a')
ax3.set_xlabel('SMA [km]')
ax3.set_ylabel('$J_a$ [rad/km] $\\times 10^{-6}$')
ax3.set_title('SMA Newton Jacobian')
ax3.legend(fontsize=8)

# (d) Eccentricity phase space in rotating frame (synthetic)
ax4 = fig.add_subplot(gs[1, :2])
t_5 = np.linspace(0, 5*T_orb_des, 5000)
Asp = J2*(RE/a_des)**2
ex_s = ey_des*np.cos(odot_des*t_5) + Asp*np.cos(mean_motion(a_des)*t_5)
ey_s = ey_des*np.sin(odot_des*t_5) + Asp*np.sin(mean_motion(a_des)*t_5)
cr   = np.cos(-odot_des*t_5)
sr   = np.sin(-odot_des*t_5)
xi_s  = ex_s*cr - ey_s*sr
eta_s = ex_s*sr + ey_s*cr
ax4.plot(xi_s*1e5, eta_s*1e5, lw=0.5, color='steelblue', alpha=0.7, label='Osc. (rotating frame)')
ax4.plot(0, e_f_des*1e5, 'r*', ms=12, label='Frozen point $(0, e_f)$')
ax4.set_xlabel('$\\xi_{rot}$ [$\\times 10^{-5}$]')
ax4.set_ylabel('$\\eta_{rot}$ [$\\times 10^{-5}$]')
ax4.set_title('5-orbit trajectory in perigee-rotating frame (synthetic)')
ax4.legend(fontsize=9)
ax4.set_aspect('equal')

# (e) Rosengren convergence (from earlier computation)
ax5 = fig.add_subplot(gs[1, 2])
ax5.semilogy(range(1, len(residuals)+1), residuals, 'o-', color='seagreen')
ax5.set_xlabel('Iteration')
ax5.set_ylabel('Residual')
ax5.set_title('Rosengren convergence')

fig.suptitle('Frozen Repeat Near-Equatorial Orbit — Summary Diagrams\n'
             f'44:3 repeat, i=10°, {alt_des:.0f} km altitude', fontsize=12)
plt.savefig('../outputs/neqfro/notebook_summary.png', dpi=130, bbox_inches='tight')
plt.show()

---
## Part 12 — Running the Full Orekit Propagation

The cells below require a working Orekit environment.
They replicate the full `design_frozen_repeat_joint.py` run interactively.

> **[Orekit required]**  Run these after executing `init_orekit()` with the data zip.

In [ ]:
# [Orekit required] — initialise JVM and load Orekit data
import pathlib, sys
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

import yaml
cfg = yaml.safe_load((ROOT / 'config.yaml').read_text())
nfg = cfg['neqfro']

from src.core.setup import init_orekit
init_orekit(str(ROOT / cfg['paths']['orekit_data']))

from org.orekit.time import AbsoluteDate
from src.core.frames import get_frames, MU as MU_OE
from src.core.frozen_orbit import (
    analytical_frozen_orbit, perigee_precession_rate, orbital_period as op_oe
)
from src.models.j3_propagator import build_orbit, build_propagator, propagate
from src.algorithms.joint_optimizer import joint_frozen_repeat

utc, gcrf, itrf, earth = get_frames()

# Parse epoch
y, mo, d = (int(x) for x in nfg['epoch'][:10].split('-'))
epoch = AbsoluteDate(y, mo, d, 0, 0, 0.0, utc)

print('Orekit initialised.')

In [ ]:
# [Orekit required] — Analytical seed
params = analytical_frozen_orbit(
    orbit_type = nfg['orbit_type'],
    k_orbits   = int(nfg['repeat_k']),
    q_days     = float(nfg['repeat_q_days']),
    i_deg      = nfg.get('inclination_deg'),
    e_init     = 0.001,
    aop_deg    = nfg['aop_deg'],
)
a_s   = params['a']
i_r   = params['i_rad']
e_s   = params['e']
raan  = np.radians(nfg['raan_deg'])
aop_r = params['aop_rad']
m0_r  = np.radians(nfg['mean_anomaly_deg'])

T_cyc   = float(nfg['repeat_q_days']) * T_SIDEREAL
omega_d = perigee_precession_rate(a_s, e_s, i_r)
dt      = float(nfg['step_size_s'])

ex_t = e_s * np.cos(aop_r)
ey_t = e_s * np.sin(aop_r)

print(f'Analytical seed: a={a_s/1e3:.3f} km  e={e_s:.4e}  aop={np.degrees(aop_r):.1f}°')

In [ ]:
# [Orekit required] — Joint iteration  (~5-15 min depending on hardware)
a_opt, ex_opt, ey_opt, history, mean_hist = joint_frozen_repeat(
    a_init    = a_s,
    ex_init   = ex_t,
    ey_init   = ey_t,
    ex_target = ex_t,
    ey_target = ey_t,
    i_rad     = i_r,
    raan_rad  = raan,
    m0_rad    = m0_r,
    epoch     = epoch,
    t_cycle   = T_cyc,
    dt        = dt,
    dt_coarse = min(dt, 60.0),
    gcrf      = gcrf,
    itrf      = itrf,
    earth     = earth,
    mu        = MU_OE,
    omega_dot = omega_d,
    n_cycles  = int(nfg.get('rosengren_n_cycles', 5)),
    k_orbits  = int(nfg['repeat_k']),
    n_iter    = int(nfg.get('joint_n_iter', 20)),
    tol_closure_deg = float(nfg.get('joint_tol_closure_deg', 0.01)),
    tol_ecc         = float(nfg.get('joint_tol_ecc', 1e-9)),
    verbose   = True,
)
e_opt   = np.hypot(ex_opt, ey_opt)
aop_opt = np.degrees(np.arctan2(ey_opt, ex_opt))
print(f'\nOptimised SMA : {a_opt:.4f} m  (Δa = {a_opt-a_s:+.4f} m)')
print(f'Optimised e   : {e_opt:.6e}  at ω = {aop_opt:.3f}°')

In [ ]:
# [Orekit required] — Convergence plots
iters     = list(range(1, len(history)+1))
closures  = [abs(h['closure_deg']) for h in history]
ecc_resid = [h['ecc_residual']     for h in history]

fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

axes[0].plot(iters, [h['a']-a_s for h in history], 'o-', color='steelblue')
axes[0].axhline(0, color='gray', lw=0.8, ls='--')
axes[0].set_ylabel('SMA correction Δa [m]')
axes[0].set_title('Joint Optimisation Convergence (Orekit J2+J3)')

axes[1].semilogy(iters, [max(c, 1e-10) for c in closures], 's-', color='tomato')
axes[1].set_ylabel('|Closure error| [deg]')

axes[2].semilogy(iters, [max(r, 1e-15) for r in ecc_resid], '^-', color='seagreen')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Ecc residual |Δ(ex,ey)|')

for ax in axes: ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---
## Summary and Reference Table

| Step | Equation | Code location |
|---|---|---|
| Frozen eccentricity | $e_f = -(J_3/2J_2)(R_E/a)\sin i$ | `frozen_orbit.frozen_eccentricity` |
| J2-corrected SMA | $n(1+\gamma) = k(\omega_E-\dot{\Omega})/q$ | `frozen_orbit.repeat_ground_track_sma` |
| J2 RAAN rate | $\dot{\Omega} = -\frac{3}{2}nJ_2(R_E/a)^2\cos i/(1-e^2)^2$ | `frozen_orbit.nodal_regression_rate` |
| J2 AoP rate | $\dot{\omega} = \frac{3}{4}nJ_2(R_E/a)^2(5\cos^2i-1)/(1-e^2)^2$ | `frozen_orbit.perigee_precession_rate` |
| Rotating frame | $\xi_{\rm rot} = e\cos(\omega - \dot{\omega}t)$ | `rosengren.eccentricity_timeseries_rotating` |
| Rosengren step | $\Delta\mathbf{z}_0 = \mathbf{z}_{\rm target} - \langle\boldsymbol{\xi}_{\rm rot}\rangle$ | `joint_optimizer.joint_frozen_repeat` |
| SMA Newton step | $\Delta a = -\Delta\lambda_{\rm AN,rad}/J_a$ | `joint_optimizer.joint_frozen_repeat` |
| SMA Jacobian | $J_a = (kT_{\rm nod}/a)(-2\dot{\Omega} - 1.5\omega_E)$ | `joint_optimizer.joint_frozen_repeat` |

### Key references

- **A34934**: Rosengren et al., *"Designing a Reference Trajectory for Frozen Repeat
  Near-Equatorial Low Earth Orbits"*, JSR 2020 — the primary reference for the
  joint iteration structure (Section IV).
- **Brouwer 1959**: *"Solution of the Problem of Artificial Satellite Theory
  Without Drag"*, AJ — secular J2 rates.
- **Coffey & Deprit 1982**: *"Third-order solution to the main problem in
  satellite theory"*, JAstron — frozen eccentricity formula.
- **D'Amico & Montenbruck 2004**: TerraSAR-X reference orbit design — repeat SMA
  with J2 corrections.